# 📌 Deep Learning Notes: Data Augmentation

> [!NOTE]
> **Summary Concept**
> **Data augmentation** is a powerful technique that artificially expands a dataset by mathematically tweaking existing data to create "new," slightly modified examples. Since Deep Learning models (like CNNs) are incredibly data-hungry, this method is a highly cost-effective way to acquire robust data, prevent memorization (overfitting), and build resilient AI without manually collecting and labeling thousands of new data points.

---

## 🗺️ Why We Use Data Augmentation (The Core Purpose)

```mermaid
graph TD
    %% styling
    classDef main fill:#e0f7fa,stroke:#00acc1,stroke-width:2px,color:#006064;
    classDef benefit fill:#e8f5e9,stroke:#43a047,stroke-width:2px,color:#1b5e20;
    classDef detail fill:#fff3e0,stroke:#fb8c00,stroke-width:1px,color:#e65100;

    A["Original Dataset<br/>(Limited & Expensive)"]:::main --> B{"Data Augmentation<br/>Engine"}:::main
    
    B --> C["Prevents Overfitting"]:::benefit
    B --> D["Improves Generalization"]:::benefit
    B --> E["Cost-Effective Scaling"]:::benefit
    B --> F["Class Balancing"]:::benefit

    C -.-> C1["Forces model to learn core concepts,<br/>not exact pixel patterns."]:::detail
    D -.-> D1["Exposes model to diverse,<br/>real-world scenarios (e.g., lighting, angles)."]:::detail
    E -.-> E1["Artificially multiplies data size<br/>without manual collection."]:::detail
    F -.-> F1["Heavily augments minority classes<br/>to eliminate bias."]:::detail
```

---

## 🛠️ Common Techniques by Data Type

### 📸 A. Computer Vision (Image Data)
Since CNNs are highly visual, image augmentation is critical for robust feature learning.
* **Geometric Transformations:**
  * **Flipping:** Horizontal (common) or vertical (rare, used in satellite/medical imagery).
  * **Rotation:** Tilting the image by a few degrees.
  * **Cropping & Zooming:** Focusing on random sections of the image.
  * **Translation:** Shifting the image left, right, up, or down.
* **Photometric Transformations (Color Space):**
  * Adjusting brightness, contrast, and saturation.
  * Converting images to grayscale.
* **Noise Injection:**
  * Adding Gaussian/static noise to simulate low-quality camera sensors and make the model robust against visual artifacts.
* **Advanced Methods:**
  * **Cutout:** Randomly dropping a black square over part of the image, forcing the model to rely on the entire object rather than one single defining feature.
  * **MixUp:** Blending two completely different images together (e.g., overlaying a dog and a cat) and assigning a blended probability (e.g., 50% dog, 50% cat).

In [ ]:
from keras import Sequential
from keras.layers import Dense, MaxPooling2D, Conv2D, Flatten
from keras.datasets import mnist


## DATA AUGMENTATION (Libraries)
- Libraries for data augmentation.
```python 
- from keras.preprocessing import image 
- from keras.preprocessing.image import ImageDataGenerator # this is depreciated in latest version because it was slow. 
- further if want to use same then can use like - 
    - from tensorflow.keras.preprocessing.image import ImageDataGenerator
```
- here we just import the keras preprocessing ImageDataGenerator for DATA AUGMENTATION.

In [ ]:
from keras.preprocessing import image   
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
import os
from pathlib import Path
curr_path = Path.cwd()
main_dirPath = curr_path.parent
dataAugment_dirPath = os.path.join(main_dirPath, 'data/augmentdata')
full_filepath = os.path.join(dataAugment_dirPath, 'cat1.jpg')
print(full_filepath)

In [ ]:
img = image.load_img(full_filepath, target_size=(200, 200))


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# import matplotlib.pyplot as plt
# plt.imshow(img)
# plt.axis('off')
# plt.show()
#  or directly we can call img - can see image without matplotlib
print(f"type of image : {type(img)}")  
# here img is not numpy array that's why img.shape will not work here. To convert it to numpy array we can use image.img_to_array() function
img


## ⚙️ Keras Image Augmentation Parameters (Deep Dive)

> [!NOTE]
> **Universal Concepts**
> These mathematical "tweaks" apply universally, whether you are using the older `ImageDataGenerator` or modern Keras preprocessing layers. They are essential for training robust Convolutional Neural Networks (CNNs).

### 1️⃣ `rotation_range` (e.g., `40`)
* **🔧 What it does:** Randomly rotates the image clockwise or counter-clockwise by any degree within the range (e.g., `-40` and `+40`).
* **🎯 Significance:** Creates **rotation invariance**. If you only train a CNN on photos of perfectly upright dogs, it will fail if a user uploads a photo of a dog lying down or if the camera was tilted. This forces the network to learn the dog's features regardless of orientation.

### 2️⃣ `shear_range` (e.g., `0.2`)
* **🔧 What it does:** Randomly applies a "shear" mapping (e.g., up to 20%). Shearing slants the image along an axis. Imagine a perfect square; if you push the top edge to the right while holding the bottom edge still, it becomes a slanted parallelogram.
* **🎯 Significance:** Simulates **perspective changes**. In the real world, we rarely look at objects perfectly straight-on. Shearing mimics looking at an object from a slight angle.

### 3️⃣ `zoom_range` (e.g., `0.2`)
* **🔧 What it does:** Randomly zooms *in* or *out* of the image. A 0.2 factor means it mathematically picks a random zoom between `[1 - 0.2, 1 + 0.2]` (80% to 120% of the original size).
* **🎯 Significance:** Creates **scale invariance**. Ensures your network can recognize an object whether it is right up close to the camera or parked far away in the background.

### 4️⃣ `horizontal_flip` (e.g., `True`)
* **🔧 What it does:** Takes roughly 50% of your images and randomly flips them backwards (like looking in a mirror).
* **🎯 Significance:** Doubles your dataset instantly for objects that are symmetrical or orientation-independent. A cat facing left is still a cat when facing right.

> [!WARNING]
> **Direction-Sensitive Data**
> Never use horizontal flipping if direction matters! For example, if you are classifying handwritten text, flipping a "b" turns it into a "d", which will negatively impact your model.

### 5️⃣ `width_shift_range` & `height_shift_range` (e.g., `0.2`)
* **🔧 What it does:** Randomly translates (slides) the image left/right or up/down by a fraction (e.g., up to 20%) of the total image width or height. When the image slides, Keras usually fills the empty gap by stretching edge pixels.
* **🎯 Significance:** Creates **translation invariance**. Forces the CNN to realize that the object does not have to be perfectly centered in the frame to be recognized.


In [ ]:
datagen = ImageDataGenerator(
    rotation_range=40, # Randomly rotates the image clockwise or counter-clockwise by any degree between -40 and +40, 
    shear_range=0.2,  # Applies a "shear" mapping (up to 20%)
    zoom_range=0.2,  # Randomly zooms in or out of the image by up to 20%    
    horizontal_flip=True, # Takes roughly 50% of your images and randomly flips them backwards and forwards horizontally
    width_shift_range=0.2, # Randomly shifts the image horizontally by up to 20%
    height_shift_range=0.2, # Randomly shifts the image vertically by up to 20%
    
)

### Another alternative way to implement the same **ImageDataGenerator**
The reason you are seeing this is that **`ImageDataGenerator` has been officially deprecated and removed** from the newest versions of Keras (Keras 3) and TensorFlow. The creators of Keras removed it because it was slow and outdated compared to modern data loading techniques.

---

### The Modern Way (Highly Recommended)

If you are learning Deep Learning right now, you should stop using `ImageDataGenerator`. The modern, industry-standard way to load and augment images is much faster because it runs on your GPU rather than your CPU.

Instead of `ImageDataGenerator`, you now do two separate things:

1. Load the data using **`image_dataset_from_directory`**.
2. Augment the data using **Keras Preprocessing Layers**.

Here is a template of how your code should look using the modern approach:

```python
import tensorflow as tf
from tensorflow.keras import layers, Sequential

# 1. Load the data (Replaces the flow_from_directory part)
train_dataset = tf.keras.utils.image_dataset_from_directory(
    'path_to_your_data/train',
    image_size=(224, 224),
    batch_size=32
)

# 2. Build an Augmentation Pipeline (Replaces the ImageDataGenerator arguments)
data_augmentation = Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.2),
])

# 3. Apply it directly inside your model!
model = Sequential([
    data_augmentation, # The augmentation happens right here!
    layers.Rescaling(1./255), # Normalizes pixel values between 0 and 1
    
    # ... then add your Conv2D layers here
    layers.Conv2D(32, (3,3), activation='relu'),
    # ...
])

```

**Why this is better:**
When you put the augmentation layers directly inside your `Sequential` model, the augmentations happen on your GPU during training (which is incredibly fast). Furthermore, when you export the model later to use in the real world, the augmentations automatically turn themselves off!

If I'd like to stick with `ImageDataGenerator` for now to finish a specific, try that `tensorflow.keras` import. 

In [ ]:
imag_array = image.img_to_array(img)
print(f"shape of image array: {imag_array.shape} \n and type of img now {type(imag_array)}")

In [ ]:
# input_batch = imag_array.reshape((1,)+imag_array.shape) or we can also use np.expand_dims() function to add an extra dimension to the array
input_batch = imag_array.reshape((1,imag_array.shape[0], imag_array.shape[1], imag_array.shape[2]))
print(f"shape of input batch : {input_batch.shape}")

**`datagen.flow()`** is an infinite generator. 
- It does not know when to stop. If you just run that line of code exactly as written, 
- it will generate augmented cats and fill up your hard drive until your computer crashes!


In [ ]:
i = 0 
savpath = os.path.join(dataAugment_dirPath, 'augGenD')
for batch in datagen.flow(input_batch, batch_size=1,save_to_dir=savpath, save_prefix='cat',save_format='jpeg'):
    # plt.figure(i)
    # imgplot = plt.imshow(image.array_to_img(batch[0]))
    i += 1
    if i == 10:
        break
